In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "bigscience/bloom-560m"

print(f"Loading {model_name}...")

# BLOOM uses its own tokenizer class but AutoTokenizer handles it fine
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# BLOOM ships with a proper <pad> token — no override needed

# Pipeline approach
# text_gen_pipeline = pipeline(
#     "text-generation",
#     model=model_name,
#     tokenizer=model_name,
#     device=0 if torch.cuda.is_available() else -1
# )

# print("\n--- Text Generation Example (Pipeline Way) ---")

# prompt_pipeline_1 = "Bratislava je hlavné mesto Slovenska a"
# result_pipeline_1 = text_gen_pipeline(prompt_pipeline_1, max_new_tokens=80, do_sample=True, temperature=0.8)
# print(f"\nPrompt: '{prompt_pipeline_1}'")
# print(f"Generated: {result_pipeline_1[0]['generated_text']}")

# prompt_pipeline_2 = "Minulý týždeň sa v parlamente hlasovalo o"
# result_pipeline_2 = text_gen_pipeline(prompt_pipeline_2, max_new_tokens=80, do_sample=True, temperature=0.8)
# print(f"\nPrompt: '{prompt_pipeline_2}'")
# print(f"Generated: {result_pipeline_2[0]['generated_text']}")

print("\n--- Manual Text Generation Example ---")

def generate_text_manual(
    prompt,
    model,
    tokenizer,
    device,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
):
    # 1. Tokenize the prompt
    # BLOOM does not expect a BOS token prepended for generation;
    # add_special_tokens=False avoids an unintended BOS prepend
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=512,
    ).to(device)

    prompt_length = inputs["input_ids"].shape[1]

    # 2. Generate tokens
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 3. Decode only newly generated tokens
    generated_ids = output_ids[0][prompt_length:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return {
        "prompt": prompt,
        "generated": generated_text,
        "full_text": prompt + generated_text,
    }

# Example 1: General Slovak knowledge prompt
# (BLOOM was trained on 46 languages including Slovak)
prompt_1 = "Bratislava je hlavné mesto Slovenska a"
result_1 = generate_text_manual(prompt_1, model, tokenizer, device)
print(f"\nPrompt: '{result_1['prompt']}'")
print(f"Generated continuation: '{result_1['generated']}'")
print(f"Full text: '{result_1['full_text']}'")

# Example 2: News summary style (lukasjanek/slovaksum domain)
prompt_2 = "Minulý týždeň sa v parlamente hlasovalo o"
result_2 = generate_text_manual(prompt_2, model, tokenizer, device)
print(f"\nPrompt: '{result_2['prompt']}'")
print(f"Generated continuation: '{result_2['generated']}'")
print(f"Full text: '{result_2['full_text']}'")

Loading bigscience/bloom-560m...


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]


--- Manual Text Generation Example ---

Prompt: 'Bratislava je hlavné mesto Slovenska a'
Generated continuation: ' šeština zadaćenja i bodoje, želite namu pravlo. Tako po vreži si pomojanjem očistiti u sevstvenom jedan jednom postoji stronika sve.
Upošalo kao dobrazovana sa korisnog opakovili naslovu pod'
Full text: 'Bratislava je hlavné mesto Slovenska a šeština zadaćenja i bodoje, želite namu pravlo. Tako po vreži si pomojanjem očistiti u sevstvenom jedan jednom postoji stronika sve.
Upošalo kao dobrazovana sa korisnog opakovili naslovu pod'

Prompt: 'Minulý týždeň sa v parlamente hlasovalo o'
Generated continuation: 'ditiť.",
'badarticleerror' => 'Adresáľadím če je moho adresa e-mailová, tak zdobanych chcete stranitevám.
Do šti nastavenie bylo tote mailovačí tlaštoje na nebo ktoru dolozku přestvojícah'
Full text: 'Minulý týždeň sa v parlamente hlasovalo oditiť.",
'badarticleerror' => 'Adresáľadím če je moho adresa e-mailová, tak zdobanych chcete stranitevám.
Do šti nastavenie bylo 

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
